# Moorea Labeled Corals EDA

Exploratory analysis of the S3-hosted Moorea Labeled Corals dataset produced by
`verify_raw_data_and_add_to_s3.py`.

All inputs are read from S3 (`s3://dev-datamermaid-sm-sources/external_validation_datasets/moorea_labeled_corals/`) — no local files needed.

In [ ]:
import json

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from mermaidseg.datasets.moorea_labeled_corals import MooreaLabeledCoralsDataset

In [ ]:
BUCKET = "dev-datamermaid-sm-sources"
PREFIX = "external_validation_datasets/moorea_labeled_corals"

s3 = boto3.client("s3")


def s3_get_json(key: str) -> dict:
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return json.loads(body)


classes = s3_get_json(f"{PREFIX}/classes.json")
colors = s3_get_json(f"{PREFIX}/colors.json")
manifest = s3_get_json(f"{PREFIX}/manifest.json")

print(f"Num classes (incl. unlabeled): {len(classes)}")
print(f"Manifest dataset: {manifest['dataset_name']} created at {manifest['created_at_utc']}")
print(
    f"Annotations parquet: s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
)

## Dataset summary (from annotations parquet)

Per-year counts are derived from the parquet on S3, augmented with the damaged-image list from `manifest.json`.

In [ ]:
from collections import defaultdict

parquet_uri = f"s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
df_meta = pd.read_parquet(parquet_uri)

years_df = (
    df_meta.groupby("year", sort=True)
    .agg(
        num_images=("image_id", "nunique"),
        num_annotations=("image_id", "size"),
        num_label_classes=("source_label_name", "nunique"),
    )
    .rename_axis("year")
)

damaged_by_year: dict[str, list[str]] = defaultdict(list)
for entry in manifest.get("damaged_images", []):
    year, _, name = entry.partition("/")
    damaged_by_year[year].append(name)
if damaged_by_year:
    years_df["num_damaged"] = years_df.index.map(lambda y: len(damaged_by_year.get(str(y), [])))

total_row = {
    "num_images": df_meta["image_id"].nunique(),
    "num_annotations": len(df_meta),
    "num_label_classes": df_meta["source_label_name"].nunique(),
}
if damaged_by_year:
    total_row["num_damaged"] = sum(len(v) for v in damaged_by_year.values())
years_df.loc["TOTAL"] = total_row
years_df

## Instantiate the dataset

In [ ]:
dataset = MooreaLabeledCoralsDataset(
    source_bucket=BUCKET,
    source_s3_prefix=PREFIX,
    annotations_path=manifest["s3"]["annotations_parquet"],
    padding=7,
)
print(f"Dataset length: {len(dataset)}")
print(f"Num source classes (incl. background): {dataset.num_source_classes}")
dataset.df_annotations.head()

## Sample image + sparse annotations per year

In [ ]:
years = sorted(dataset.df_images["year"].unique())
n = len(years)
ncols = min(3, n) if n > 0 else 1
nrows = int(np.ceil(n / ncols)) if n > 0 else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

rng = np.random.default_rng(0)
for ax, year in zip(axes.ravel(), years, strict=False):
    year_rows = dataset.df_images.index[dataset.df_images["year"] == year].to_numpy()
    if len(year_rows) == 0:
        ax.axis("off")
        ax.set_title(f"{year}: no images")
        continue
    idx = int(rng.choice(year_rows))
    image_id = dataset.df_images.loc[idx, "image_id"]
    image_ext = dataset.df_images.loc[idx, "image_ext"]
    image = dataset.read_image(image_id=image_id, year=year, image_ext=image_ext)

    anns = dataset.df_annotations.loc[
        (dataset.df_annotations["image_id"] == image_id) & (dataset.df_annotations["year"] == year),
        ["row", "col", "source_label_name"],
    ]

    ax.imshow(image)
    seen_labels = []
    for _, r in anns.iterrows():
        rgb = colors.get(r["source_label_name"], [255, 0, 0])
        ax.scatter(
            r["col"],
            r["row"],
            color=np.array(rgb) / 255.0,
            s=30,
            edgecolors="white",
            linewidths=0.4,
        )
        seen_labels.append(r["source_label_name"])
    handles = [
        Patch(facecolor=np.array(colors.get(name, [255, 0, 0])) / 255.0, label=name)
        for name in sorted(set(seen_labels))[:10]
    ]
    ax.set_title(f"{year} / {image_id} ({len(anns)} pts)")
    ax.axis("off")
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=6, framealpha=0.7)

for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Top-20 label classes per year

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
for ax, year in zip(axes.ravel(), years, strict=False):
    counts = (
        dataset.df_annotations.loc[dataset.df_annotations["year"] == year, "source_label_name"]
        .value_counts()
        .head(20)
    )
    if counts.empty:
        ax.set_title(f"{year}: no annotations")
        ax.axis("off")
        continue
    bar_colors = [np.array(colors.get(name, [128, 128, 128])) / 255.0 for name in counts.index]
    ax.barh(range(len(counts)), counts.to_numpy()[::-1], color=bar_colors[::-1])
    ax.set_yticks(range(len(counts)))
    ax.set_yticklabels(counts.index[::-1], fontsize=8)
    year_mask = dataset.df_annotations["year"] == year
    n_unique = dataset.df_annotations.loc[year_mask, "source_label_name"].nunique()
    ax.set_title(f"{year} (top 20 of {n_unique})")
for ax in axes.ravel()[len(years) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Sanity-check `__getitem__`

Confirm that `dataset[idx]` returns a `(image, source_labels)` pair where the source-labels mask is rendered consistently with `BaseCoralDataset` semantics.

In [ ]:
idx = 0
image, source_mask = dataset[idx]
row = dataset.df_images.loc[idx]
print(f"Image idx={idx}: year={row['year']}, image_id={row['image_id']}{row['image_ext']}")
print(f"Image shape: {image.shape}, mask shape: {source_mask.shape}")
print(
    f"Unique mask ids (excl. 0): {sorted({int(v) for v in np.unique(source_mask) if v != 0})[:20]}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")

vis = np.zeros((*source_mask.shape, 3), dtype=np.uint8)
for local_id, name in dataset.source_id2name.items():
    rgb = colors.get(name, [255, 0, 0])
    vis[source_mask == local_id] = np.array(rgb, dtype=np.uint8)

axes[1].imshow(image)
axes[1].imshow(vis, alpha=0.6)
axes[1].set_title("Source-label mask overlay")
axes[1].axis("off")
plt.tight_layout()
plt.show()